# Transformación Silver con publicaciones únicas

Esta es la versión operativa. El notebook 01 conserva las salidas históricas de la primera ejecución. Se reconstruyen solo las tablas Silver intermedias: publicaciones, mediciones válidas y rechazados. `mediciones_vigentes` queda reservada para la actualización incremental posterior.


In [1]:
import json
import sys
from pathlib import Path
from pyspark.sql import functions as F

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config' / 'silver.json').is_file()), None)
if project_root is None:
    raise FileNotFoundError('No se encuentra config/silver.json')
sys.path.insert(0, str(project_root / 'scripts'))
from silver_quality import transform_publicaciones, transform_mediciones
silver = json.loads((project_root / 'config' / 'silver.json').read_text(encoding='utf-8'))
bronze = json.loads((project_root / silver['bronze_config']).read_text(encoding='utf-8'))
print('Origen:', bronze['tables'])
print('Tablas intermedias:', {name: silver['tables'][name] for name in ('publicaciones', 'mediciones', 'rechazados')})
print('Tabla definitiva reservada:', silver['tables']['mediciones_vigentes'])


Origen: {'publicaciones': 'workspace.lensdata_bronze.publicaciones', 'mediciones': 'workspace.lensdata_bronze.mediciones'}
Tablas intermedias: {'publicaciones': 'workspace.lensdata_silver.publicaciones', 'mediciones': 'workspace.lensdata_silver.mediciones', 'rechazados': 'workspace.lensdata_silver.rechazados'}
Tabla definitiva reservada: workspace.lensdata_silver.mediciones_vigentes


## Validación y separación de registros

Las conversiones seguras convierten entradas no válidas en `NULL` y registran el motivo. Bronze debe permanecer como evidencia original para futuras comparaciones o cambios en la estructura de las capas posteriores.


In [2]:
raw_publicaciones = spark.table(bronze['tables']['publicaciones'])
raw_mediciones = spark.table(bronze['tables']['mediciones'])
publicaciones_ok, publicaciones_rechazadas, publicaciones_conflictivas = transform_publicaciones(raw_publicaciones, silver)
mediciones_ok, mediciones_rechazadas = transform_mediciones(raw_mediciones, publicaciones_ok, publicaciones_conflictivas, silver)
rechazados = publicaciones_rechazadas.unionByName(mediciones_rechazadas)

count_publicaciones = raw_publicaciones.count()
count_mediciones = raw_mediciones.count()
assert publicaciones_ok.count() + publicaciones_rechazadas.count() == count_publicaciones
assert mediciones_ok.count() + mediciones_rechazadas.count() == count_mediciones
print('Publicaciones Bronze / válidas / rechazadas:', count_publicaciones, publicaciones_ok.count(), publicaciones_rechazadas.count())
print('Mediciones Bronze / válidas / rechazadas:', count_mediciones, mediciones_ok.count(), mediciones_rechazadas.count())
display(rechazados.withColumn('reason_code', F.explode('reason_codes')).groupBy('entity', 'reason_code').count())

assert publicaciones_ok.count() == publicaciones_ok.select('plataforma', 'video_id').distinct().count()
print('Claves de publicaciones conflictivas:', publicaciones_conflictivas.count())


Publicaciones Bronze / válidas / rechazadas: 40 40 0
Mediciones Bronze / válidas / rechazadas: 473 469 4


,entity,reason_code,count
0,mediciones,UNKNOWN_VIDEO,1
1,mediciones,NEGATIVE_METRIC,1
2,mediciones,REQUIRED_FIELD,1
3,mediciones,INVALID_NUMBER,1


Claves de publicaciones conflictivas: 0


## Escritura de tablas intermedias

Se reconstruyen publicaciones, mediciones válidas y rechazados. La tabla definitiva `mediciones_vigentes` no se crea ni se sobrescribe aquí.


In [3]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze['catalog']}.{silver['schema']}")
for dataframe, table in ((publicaciones_ok, silver['tables']['publicaciones']),
                         (mediciones_ok, silver['tables']['mediciones']),
                         (rechazados, silver['tables']['rechazados'])):
    (dataframe.write.format('delta').mode('overwrite')
     .option('overwriteSchema', 'true').saveAsTable(table))
print('Tablas Silver intermedias reconstruidas:', [silver['tables'][name] for name in ('publicaciones', 'mediciones', 'rechazados')])


Tablas Silver intermedias reconstruidas: ['workspace.lensdata_silver.publicaciones', 'workspace.lensdata_silver.mediciones', 'workspace.lensdata_silver.rechazados']
